# FAKTA — LSTM Training
Train the BiLSTM hoax classifier.

In [ ]:
import sys
sys.path.insert(0, 'src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Load dataset
df = pd.read_csv('data/training/combined.csv')
print(f"Dataset: {len(df)} samples")
print(df['label'].value_counts())

In [ ]:
# Preprocessing
from preprocessing.cleaning import clean_text
from preprocessing.slang_normalizer import normalize_slang
from preprocessing.feature_extractor import extract_features

# Clean and normalize
df['cleaned'] = df['text'].apply(clean_text)
df['normalized'] = df['cleaned'].apply(normalize_slang)
print(f"Sample cleaned text:\n{df['normalized'].iloc[0][:200]}")

In [ ]:
# Extract linguistic features
features_df = pd.DataFrame([extract_features(t).to_dict() for t in df['normalized']])
print(f"Extracted {len(features_df.columns)} linguistic features")
print(features_df.head())

In [ ]:
# Tokenize and pad sequences
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

max_words = 20000
max_len = 200

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['normalized'].tolist())

sequences = tokenizer.texts_to_sequences(df['normalized'].tolist())
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

# Encode labels
label_map = {'valid': 0, 'hoax': 1, 'uncertain': 2}
y = np.array([label_map[l] for l in df['label']])

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Vocabulary size: {len(tokenizer.word_index)}")

In [ ]:
# Train/val/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.18, random_state=42, stratify=y_train
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
# Class weights
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))
print(f"Class weights: {class_weight}")

In [ ]:
# Build and train model
sys.path.insert(0, 'src/classifier')
from lstm_model import build_lstm_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

model = build_lstm_model(
    max_words=max_words,
    max_len=max_len,
    embedding_dim=128,
    lstm_units=64,
    dropout_rate=0.3,
)

model.summary()

In [ ]:
# Train
os.makedirs('models/lstm', exist_ok=True)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('models/lstm/lstm_model.keras', save_best_only=True, monitor='val_loss'),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks,
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.4f}, Test Loss: {loss:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)
labels = ['valid', 'hoax', 'uncertain']
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=labels))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Save tokenizer
with open('models/lstm/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved to models/lstm/tokenizer.pkl")